# Deep Rotated MNIST: first versus last layers

Load `deep-25-epoch-model.pkl` without retraining. Compare the first and last **post-ReLU hidden layers** within the shared backbone and each of the digit, parity, and rotation branches. These are layers 0 and 4, not the final prediction layers. All comparisons use exactly the same test images and rotations.

The checkpoint predates the rotation-loss fixes. Similarity describes this saved model's representations; it does not establish good rotation prediction or task-specific tuning.


In [1]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

# Run from either the repository root or the rotated-MNIST directory.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "model.py").exists():
    PROJECT_DIR = PROJECT_DIR / "rotated-MNIST"
if not (PROJECT_DIR / "model.py").exists():
    raise FileNotFoundError("Run from the repository root or rotated-MNIST directory")
sys.path.insert(0, str(PROJECT_DIR))
from model import MultiTaskDNN
from data_setup import create_dataloaders

SEED = 67
N_SAMPLES = 1000  # RSA uses N * (N - 1) / 2 pairwise distances.
BATCH_SIZE = 128
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Load the saved deep model

The state dictionary matches five shared blocks and five hidden blocks per task, all width 16. Inference runs on CPU in evaluation mode, using stored batch-normalization statistics. No optimizer or training loop is created.


In [ ]:
model_cfg = {
    "input_shape": 784,
    "shared_layers": 5,
    "task_layers": 5,
    "shared_nodes": 16,
    "task_nodes": 16,
    "digit_out": 10,
    "parity_out": 2,
    "rotation_out": 1,
}
checkpoint_path = PROJECT_DIR / "deep-25-epoch-model.pkl"
model = MultiTaskDNN(model_cfg, {"activation": "relu"})
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()
print(f"Loaded {checkpoint_path.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## Collect matched activations

Use the first 1,000 held-out MNIST images in dataset order. Each image receives one random rotation during this single collection pass. Resetting the seed immediately before collection reproduces the rotations when this cell is rerun in the same environment. Cached activations are reused by every metric; no metric resamples the data.


In [ ]:
data_cfg = {
    "dataset_path": str(PROJECT_DIR / "DNN_Data"),
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "max_digit": 9,
    "max_rotation_degrees": 360,
}
_, test_loader = create_dataloaders(data_cfg)
comparisons = {
    branch: (f"{branch}_Layer_0", f"{branch}_Layer_4")
    for branch in ["Shared", "Digit", "Parity", "Rotation"]
}
layer_names = [name for pair in comparisons.values() for name in pair]
activation_batches = {name: [] for name in layer_names}
digit_batches, angle_batches = [], []
collected = 0
random.seed(SEED)
torch.manual_seed(SEED)
model.eval()
with torch.no_grad():
    for images, digits, _, angle_deg in test_loader:
        take = min(N_SAMPLES - collected, len(images))
        _, _, _, acts = model(images[:take])
        for name in layer_names:
            activation_batches[name].append(acts[name].cpu())
        digit_batches.append(digits[:take])
        angle_batches.append(angle_deg[:take])
        collected += take
        if collected >= N_SAMPLES:
            break

activations = {
    name: torch.cat(batches).numpy().astype(np.float64)
    for name, batches in activation_batches.items()
}
digits = torch.cat(digit_batches).numpy()
angles_deg = torch.cat(angle_batches).numpy()
assert all(a.shape == (collected, 16) for a in activations.values())
print(f"Collected {collected} matched test examples across {len(activations)} layers")
display(pd.DataFrame({"digit": digits}).value_counts().sort_index())


## Similarity metrics

- **Procrustes similarity:** center both matrices, normalize each by its Frobenius norm, and sum the singular values of their cross-product. This measures alignment after an optimal orthogonal transformation (including reflection) and uniform scaling. Higher is more similar, from 0 to 1. Also report SciPy-style disparity, `1 - similarity?`, where lower is better.
- **Linear CKA:** squared Frobenius norm of the centered cross-product divided by the corresponding within-layer norms. Higher is more similar, from 0 to 1. This is the standard biased empirical estimator, without a finite-sample correction.
- **RSA:** Spearman correlation between the upper triangles of the two Euclidean representational distance matrices. Each distance compares a pair of input images within one layer. Higher means more similar distance rankings; the range is -1 to 1. Units are not standardized individually.

Undefined metrics for constant representations are reported as NaN. These scores quantify different aspects of similarity and should not be interpreted as interchangeable accuracy measurements.


In [ ]:
def centered(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError("Expected a finite samples-by-neurons matrix")
    return x - x.mean(axis=0, keepdims=True)


def procrustes_similarity(x, y):
    x, y = centered(x), centered(y)
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    if denom == 0:
        return np.nan
    return float(np.clip(np.linalg.svd(x.T @ y, compute_uv=False).sum() / denom, 0, 1))


def linear_cka(x, y):
    x, y = centered(x), centered(y)
    denom = np.linalg.norm(x.T @ x, "fro") * np.linalg.norm(y.T @ y, "fro")
    if denom == 0:
        return np.nan
    return float(np.clip(np.linalg.norm(x.T @ y, "fro") ** 2 / denom, 0, 1))


def rsa_similarity(x, y):
    dx, dy = pdist(x, metric="euclidean"), pdist(y, metric="euclidean")
    if np.ptp(dx) == 0 or np.ptp(dy) == 0:
        return np.nan
    return float(spearmanr(dx, dy).statistic)


def compare_layers(x, y):
    if x.shape[0] != y.shape[0] or x.shape[0] < 3:
        raise ValueError("Need at least three matched examples")
    proc = procrustes_similarity(x, y)
    return {
        "Procrustes similarity": proc,
        "Procrustes disparity": 1 - proc ** 2,
        "Linear CKA": linear_cka(x, y),
        "RSA (Spearman)": rsa_similarity(x, y),
    }


In [ ]:
# Check identity and invariance to translation, uniform scale, and rotation.
rng = np.random.default_rng(SEED)
x_check = rng.normal(size=(40, 16))
q, _ = np.linalg.qr(rng.normal(size=(16, 16)))
y_check = 3 * x_check @ q + rng.normal(size=16)
for metric in [procrustes_similarity, linear_cka, rsa_similarity]:
    assert np.isclose(metric(x_check, x_check), 1)
    assert np.isclose(metric(x_check, y_check), 1)
assert np.isnan(linear_cka(np.ones((10, 16)), np.ones((10, 16))))
print("Metric sanity checks passed")


In [ ]:
rows = []
for branch, (first, last) in comparisons.items():
    rows.append({
        "Branch": branch,
        "First layer": first,
        "Last layer": last,
        **compare_layers(activations[first], activations[last]),
    })
results = pd.DataFrame(rows).set_index("Branch")
display(results.round(4))

scores = ["Procrustes similarity", "Linear CKA", "RSA (Spearman)"]
ax = results[scores].plot.bar(figsize=(10, 4), ylim=(-1, 1.05), rot=0)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_ylabel("Similarity")
ax.set_title("Deep checkpoint: first versus last hidden layer")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## Inspect representational distances

Show a small matched subset to inspect the geometry underlying RSA. Rows and columns represent the same test examples in both panels. Each pair of panels shares a color scale; RSA itself is computed above using all collected examples.


In [ ]:
N_DISPLAY = min(60, collected)
fig, axes = plt.subplots(len(comparisons), 2, figsize=(10, 16))
for row, (branch, pair) in enumerate(comparisons.items()):
    rdms = [squareform(pdist(activations[name][:N_DISPLAY])) for name in pair]
    vmax = max(rdm.max() for rdm in rdms)
    for col, (name, rdm) in enumerate(zip(pair, rdms)):
        ax = axes[row, col]
        im = ax.imshow(rdm, cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(name)
        ax.set_xlabel("Test example")
        ax.set_ylabel("Test example")
    fig.colorbar(im, ax=list(axes[row]), label="Euclidean distance", shrink=0.8)
fig.suptitle("Matched representational distance matrices", y=0.995)
plt.show()


## Reading the results

Higher similarity means that the first and last hidden layers preserve more of the same geometry under the chosen metric. Lower similarity indicates a larger transformation; it does not by itself indicate better learning. High RSA can coexist with lower CKA because RSA compares distance ranks rather than their magnitudes.

Treat these as descriptive results for one checkpoint and one test sample. The four comparisons cover different paths: the shared comparison spans the backbone, while each task comparison spans its branch. Depth, ReLU sparsity, the narrow 16-neuron representation, and the original training objective can all affect the scores. Establishing task selectivity would require additional analyses using digit, parity, and rotation labels. No statistical independence or significance is assumed for the pairwise distances.
